## Import the working libraries 

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from itertools import product

import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("✓ Libraries loaded for segmentation analysis")

✓ Libraries loaded for segmentation analysis


In [2]:
# Load cleaned dataset
df = pd.read_csv("cleaned_insurance_data.csv")

In [3]:
df.head()

,policy_id,subscription_length,vehicle_age,customer_age,region_code,region_density,segment,model,fuel_type,max_torque,max_power,engine_type,airbags,is_esc,is_adjustable_steering,is_tpms,is_parking_sensors,is_parking_camera,rear_brakes_type,displacement,cylinder,transmission_type,steering_type,turning_radius,length,width,gross_weight,is_front_fog_lights,is_rear_window_wiper,is_rear_window_washer,is_rear_window_defogger,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,claim_status
0,POL045360,9.3000,1.2000,41,C8,8794,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,1.5 L U2 CRDi,6,1,1,1,1,1,Disc,1493,4,Automatic,Power,5.2000,4300,1790,1720,1,1,1,1,1,1,1,1,1,0,1,1,3,0
1,POL016745,8.2000,1.8000,35,C2,27003,C1,M9,Diesel,200Nm@1750rpm,97.89bhp@3600rpm,i-DTEC,2,0,1,0,1,1,Drum,1498,4,Manual,Electric,4.9000,3995,1695,1051,1,0,0,1,0,1,1,1,1,1,1,1,4,0
2,POL007194,9.5000,0.2000,44,C8,8794,C2,M4,Diesel,250Nm@2750rpm,113.45bhp@4000rpm,1.5 L U2 CRDi,6,1,1,1,1,1,Disc,1493,4,Automatic,Power,5.2000,4300,1790,1720,1,1,1,1,1,1,1,1,1,0,1,1,3,0
3,POL018146,5.2000,0.4000,44,C10,73430,A,M1,CNG,60Nm@3500rpm,40.36bhp@6000rpm,F8D Petrol Engine,2,0,0,0,1,0,Drum,796,3,Manual,Power,4.6000,3445,1515,1185,0,0,0,0,0,0,0,1,0,0,0,1,0,0
4,POL049011,10.1000,1.0000,56,C13,5410,B2,M5,Diesel,200Nm@3000rpm,88.77bhp@4000rpm,1.5 Turbocharged Revotorq,2,0,1,0,1,0,Drum,1497,4,Manual,Electric,5.0000,3990,1755,1490,0,0,0,0,0,1,1,1,0,0,1,1,5,0


In [ ]:
# Drop PIFs
df = df.drop(columns=["policy_id"])

In [ ]:
# Define target
X = df.drop("claim_status", axis=1)
y = df["claim_status"]

In [ ]:
# Feature grouping
numerical_features = [
    "subscription_length",
    "vehicle_age",
    "customer_age",
    "region_density",
    "displacement",
    "cylinder",
    "turning_radius",
    "length",
    "width",
    "gross_weight",
    "ncap_rating"
]

categorical_features = [
    "region_code",
    "segment",
    "model",
    "fuel_type",
    "engine_type",
    "transmission_type",
    "steering_type",
    "rear_brakes_type"
]

# Binary features
binary_features = [col for col in X.columns if col not in numerical_features + categorical_features]

In [ ]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("bin", "passthrough", binary_features)
    ]
)


In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.pipeline import Pipeline

#### Logistic Regression

In [ ]:
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

#### Random Forest

In [ ]:
rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

#### XGBoost

In [ ]:
xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

#### LightGBM

In [ ]:
lgbm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    ))
])

#### Store models for iteration

In [ ]:
models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm
}

#### Train all models

In [ ]:
for name, model in models.items():
    print(f"\nTraining: {name}")
    model.fit(X_train, y_train)

#### Evaluation function

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

def evaluate_model(model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "log_loss": log_loss(y_test, y_proba)
    }

In [ ]:
results = []

for name, model in models.items():
    
    print(f"Evaluating: {name}")
    
    metrics = evaluate_model(model, X_test, y_test)
    metrics["model"] = name
    
    results.append(metrics)

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "log_loss"]
]

results_df = results_df.sort_values(by="roc_auc", ascending=False)

results_df

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

print("Best Model:", best_model_name)